In [1]:
# ============================================================================
# LINGALA MEDICAL TRIAGE SYSTEM - ENHANCED VERSION
# Transformer Models + External Evaluation
# ============================================================================
# Models implemented:
# 1. Model 2: Fine-Tuned XLM-RoBERTa (Enhanced)
# 2. Model 3: Cross-Lingual Transfer (French → Lingala) - Enhanced
# 3. Model 4: Fine-Tuned AfriBERTa (Enhanced)
# 4. Baseline: SVM + TF-IDF with character n-grams
# 
# ENHANCEMENTS:
# - Improved SVM with character n-grams and proper regularization
# - Ensemble model combining multiple approaches
# - Confidence-based prediction with rejection option
# - Domain adaptation with pseudo-labeling
# - Enhanced cross-lingual transfer with curriculum learning
# ============================================================================

import os
import csv
import json
import random
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils import resample

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EvalPrediction
)
from torch.utils.data import Dataset

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Create results directory
os.makedirs("./results", exist_ok=True)
os.makedirs("./models", exist_ok=True)

# Label mappings
LABEL2ID = {"Emergency": 0, "Moderate": 1, "Low": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = ["Emergency", "Moderate", "Low"]

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("✅ Setup complete")

PyTorch version: 2.9.1+cu128
CUDA available: False
✅ Setup complete


In [2]:
# ============================================================================
# SECTION 2: ENHANCED STOPWORD HANDLING
# ============================================================================

from lingala_stopwords import remove_stopwords, tokenize_and_filter, STOPWORDS_MEDICAL

print(f"Loaded {len(STOPWORDS_MEDICAL)} medical stopwords")
print("Sample stopwords:", list(STOPWORDS_MEDICAL)[:20])

Loaded 130 medical stopwords
Sample stopwords: ['wana', 'a', 'nani', 'solo', 'nao', 'biso', 'e', 'mpe', 'wapi', 'nse', 'moko', 'hm', 'nanu', 'na', 'likoló', 'i', 'bayali', 'kuna', 'ozo', 'ozali']


In [3]:
# ============================================================================
# SECTION 3: DATA LOADING AND PREPROCESSING (ENHANCED)
# ============================================================================

CSV_PATH = "lingala_triage_final_616.csv"

def load_dataset(csv_path):
    samples = []
    with open(csv_path, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = row['text'].strip()
            label = row['urgency'].strip()
            if text and label in LABEL2ID:
                # Clean text: normalize whitespace, remove excessive punctuation
                text = re.sub(r'\s+', ' ', text)
                text = re.sub(r'[!?]+', '!', text)
                samples.append({'text': text, 'label': label})
    return samples

def stratified_split(samples, train_ratio=0.7, val_ratio=0.15):
    by_label = defaultdict(list)
    for s in samples:
        by_label[s['label']].append(s)
    
    train, val, test = [], [], []
    for label, items in by_label.items():
        random.shuffle(items)
        n = len(items)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        train.extend(items[:n_train])
        val.extend(items[n_train:n_train + n_val])
        test.extend(items[n_train + n_val:])
    
    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)
    return train, val, test

# Load and split
all_samples = load_dataset(CSV_PATH)
print(f"Loaded {len(all_samples)} samples")
print(f"Distribution: {Counter(s['label'] for s in all_samples)}")

train_samples, val_samples, test_samples = stratified_split(all_samples)
print(f"\nTrain: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")
print(f"Train distribution: {Counter(s['label'] for s in train_samples)}")

Loaded 616 samples
Distribution: Counter({'Low': 450, 'Moderate': 107, 'Emergency': 59})

Train: 430, Val: 91, Test: 95
Train distribution: Counter({'Low': 315, 'Moderate': 74, 'Emergency': 41})


In [5]:
import random
import re
import json
import csv
import os
from collections import Counter

# ── Load original Emergency samples ──────────────────────────────────────────
with open("lingala_triage_final_616.csv", encoding="utf-8-sig") as f:
    all_rows = list(csv.DictReader(f))

emergency_texts = [
    r["text"].strip()
    for r in all_rows
    if r["urgency"].strip() == "Emergency"
]
print(f"\nOriginal Emergency samples: {len(emergency_texts)}")

# Quick content-word preview
print("Sample content tokens after stopword removal:")
for ex in emergency_texts[:3]:
    tokens = tokenize_and_filter(ex, STOPWORDS_MEDICAL)
    print(f"  {tokens}")


# ── Quality filter ────────────────────────────────────────────────────────────
MIN_CONTENT_TOKENS = 3   # discard augmented samples below this threshold

def has_enough_content(text: str) -> bool:
    """Return True if text has >= MIN_CONTENT_TOKENS after stopword removal."""
    return len(tokenize_and_filter(text, STOPWORDS_MEDICAL)) >= MIN_CONTENT_TOKENS


# ── 1. Synonym substitution (stopword-aware) ─────────────────────────────────
# Only clinical/content words are swapped — stopwords are left untouched.
# This avoids creating semantically empty augmentations.
SYNONYMS = {
    # Severity / intensity
    "makasi":               ["mingi", "ya ndelo", "koleka ndelo", "ya nkanda"],
    "mpasi":                ["pasi", "mawa ya nzoto", "mpasi ya ndelo", "pasi monene"],
    "mingi":                ["makasi", "lisusu", "koleka", "mpenza"],
    "ekomi":                ["ekomaki", "ezali", "ebandi", "ekotaki"],
    # Deterioration
    "elengelemi lisusu te": ["elingaki lisusu te", "ekufi", "ebebisami", "etikali te"],
    "ekomi kobeba":         ["ekomaki kobeba", "ezali mabe", "ebandaki kobeba"],
    "nzoto ekufi":          ["nzoto elingaki", "nzoto ezali pasi", "nzoto ebebisami"],
    # Emotional/urgency
    "Nazali kolela":        ["Nakobanga mingi", "Nazali na bobangi", "Nazali na mawa"],
    "Mpasi ezali":          ["Mawa ya nzoto ezali", "Pasi ezali", "Mpasi monene ezali"],
    "Bobangi ekomi":        ["Nsomo ekomi", "Kobanga ekomi", "Mawa ekomi"],
    "Mawa ya nzoto ekomi":  ["Mpasi ya nzoto ekomi", "Bobangi ya nzoto ekomi"],
    "Nazali na mawa mingi": ["Nazali kobanga mingi", "Nzoto na ngai ezali na pasi"],
    # Body responses
    "kolumba":              ["kokweya", "kolobana", "kovirika"],
    "kobeta motema":        ["kokangama motema", "motema kobeta nokinoki"],
    "kopema":               ["kozwa mpema", "kobeta souffle"],
    # Help-seeking
    "nalingi lisungi":      ["nalingi lisaidi", "nazali kotela lisungi", "nasengi lisungi"],
    "nalingi lisalisi":      ["nalingi lisungi", "nasengi lisalisi", "nazali kotela lisalisi"],
    "nakoka lisusu te":     ["nakoki lisusu te", "nzoto elingaka lisusu te", "nazali na makasi te"],
}

# Ensure no synonym key is a stopword (would never match in content)
_stopword_keys = [k for k in SYNONYMS if k.lower() in STOPWORDS_MEDICAL]
if _stopword_keys:
    print(f"  ⚠ Synonym keys that are stopwords (will rarely match): {_stopword_keys}")

def synonym_substitution(text: str, n_swaps: int = 2) -> str:
    """
    Swap up to n_swaps clinical phrases with synonyms.
    Skips any token that is a pure stopword.
    """
    keys = list(SYNONYMS.keys())
    random.shuffle(keys)
    result  = text
    swapped = 0
    for key in keys:
        if swapped >= n_swaps:
            break
        # Only swap if key is a content phrase (not a pure stopword)
        if key.lower() in STOPWORDS_MEDICAL:
            continue
        if key.lower() in result.lower():
            replacement = random.choice(SYNONYMS[key])
            result  = re.sub(re.escape(key), replacement, result, count=1, flags=re.IGNORECASE)
            swapped += 1
    return result


# ── 2. Sentence reordering ────────────────────────────────────────────────────
def reorder_sentences(text: str) -> str:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(parts) >= 2:
        parts[0], parts[1] = parts[1], parts[0]
    return " ".join(parts)


# ── 3. Prefix injection ───────────────────────────────────────────────────────
URGENT_PREFIXES = [
    "Nasengi lisungi ya mbala moko pamba te",
    "Nsomo na ngai ekomi makasi koleka",
    "Nzoto na ngai ezali kokweya, nalingi lisungi",
    "Situation na ngai ezali ya likama",
    "Nakosepela soki ozali kosalisa ngai mbala moko",
    "Nazali kofanda malamu te",
    "nalingi lisungi ya mbango",
    "Mpasi ezali koleka ndelo",
    "tosungami",
    "Tango ezali kokufa",
    "nalingi mosungi",
]

def inject_prefix(text: str) -> str:
    return f"{random.choice(URGENT_PREFIXES)}. {text}"


# ── 4. Claude API paraphrase ──────────────────────────────────────────────────
def paraphrase_with_claude_sync(texts, n_per_text=2):
    import requests
    results    = []
    batch_size = 5

    for i in range(0, len(texts), batch_size):
        batch    = texts[i:i+batch_size]
        numbered = "\n".join(f"{j+1}. {t}" for j, t in enumerate(batch))
        prompt   = (
            "Tu es un expert en langue Lingala et en terminologie médicale d'urgence.\n\n"
            f"Voici {len(batch)} phrases de patients en Lingala décrivant des symptômes d'urgence médicale.\n\n"
            f"Pour chaque phrase, génère exactement {n_per_text} paraphrases en Lingala qui :\n"
            "- Conservent le sens d'urgence médicale\n"
            "- Utilisent des mots différents mais restent naturelles en Lingala\n"
            "- Gardent le registre de patient qui souffre\n\n"
            "Réponds UNIQUEMENT en JSON valide, sans markdown ni backticks, format :\n"
            '{"1": ["paraphrase_a", "paraphrase_b"], "2": ["paraphrase_a", "paraphrase_b"]}\n\n'
            f"Phrases :\n{numbered}"
        )
        try:
            resp = requests.post(
                "https://api.anthropic.com/v1/messages",
                headers={
                    "Content-Type": "application/json",
                    "X-API-Key": os.environ.get("ANTHROPIC_API_KEY") # Use environment variable
                },
                json={
                    "model":      "claude-sonnet-4-20250514",
                    "max_tokens": 1500,
                    "messages":   [{"role": "user", "content": prompt}],
                },
                timeout=60,
            )
            resp.raise_for_status()
            raw  = re.sub(r"^```json\s*|```$", "", resp.json()["content"][0]["text"].strip(), flags=re.MULTILINE).strip()
            data = json.loads(raw)
            for key in sorted(data.keys(), key=lambda x: int(x)):
                results.extend(data[key])
            print(f"  ✓ Batch {i//batch_size+1}: {len(batch)} texts → {sum(len(v) for v in data.values())} paraphrases")
        except Exception as e:
            print(f"  ✗ Batch {i//batch_size+1} failed ({e}) — synonym fallback")
            for t in batch:
                results.extend([synonym_substitution(t) for _ in range(n_per_text)])
    return results


# ── Run augmentation ──────────────────────────────────────────────────────────
print("\nAugmenting Emergency samples...")
augmented_emergency = []

for text in emergency_texts:
    augmented_emergency.append(synonym_substitution(text, n_swaps=1))
    augmented_emergency.append(synonym_substitution(text, n_swaps=2))
    augmented_emergency.append(reorder_sentences(text))
    augmented_emergency.append(inject_prefix(text))

print(f"  Rule-based: +{len(augmented_emergency)} samples")

print("  Calling Claude API for paraphrases (2 per original)...")
claude_paraphrases = paraphrase_with_claude_sync(emergency_texts, n_per_text=2)
augmented_emergency.extend(claude_paraphrases)
print(f"  Claude paraphrases: +{len(claude_paraphrases)} samples")

# ── Deduplicate ───────────────────────────────────────────────────────────────
seen   = set()
deduped = []
for t in augmented_emergency:
    if t not in seen:
        seen.add(t)
        deduped.append(t)
print(f"  After dedup: {len(deduped)} samples")

# ── Quality filter (stopword-aware) ──────────────────────────────────────────
before_filter = len(deduped)
deduped = [t for t in deduped if has_enough_content(t)]
removed = before_filter - len(deduped)
print(f"  After quality filter (<{MIN_CONTENT_TOKENS} content tokens removed): "
      f"{len(deduped)} samples  [{removed} discarded]")

augmented_emergency = deduped
augmented_samples   = [{"text": t, "label": "Emergency"} for t in augmented_emergency]

# ── Rebuild augmented train split ─────────────────────────────────────────────
# train_samples must already be defined from cell_load_dataset
orig_emergency_train = [s for s in train_samples if s["label"] == "Emergency"]
non_emergency_train  = [s for s in train_samples if s["label"] != "Emergency"]

train_samples_aug = non_emergency_train + orig_emergency_train + augmented_samples
random.shuffle(train_samples_aug)

print(f"\nFinal augmented train set: {len(train_samples_aug)} samples")
print("Distribution:", Counter(s["label"] for s in train_samples_aug))

# ── Recompute class weights from augmented distribution ───────────────────────
label_order   = ["Emergency", "Moderate", "Low"]
label_counts  = Counter(s["label"] for s in train_samples_aug)
total         = sum(label_counts.values())
class_weights = []
for l in label_order:
    count = label_counts.get(l, 0)
    if count == 0:
        class_weights.append(0.0) # Assign 0 weight if no samples for this class
    else:
        class_weights.append(round(total / (len(label_order) * count), 4))
print(f"\nUpdated class weights {label_order}: {class_weights}")
print("\nVal/Test sets UNCHANGED (no augmentation on val/test).")
print(f"  Val  : {len(val_samples)}  {Counter(s['label'] for s in val_samples)}")
print(f"  Test : {len(test_samples)}  {Counter(s['label'] for s in test_samples)}")

# ── Save augmented train to disk ──────────────────────────────────────────────
os.makedirs("./results", exist_ok=True)
with open("./results/augmented_train.json", "w", encoding="utf-8") as f:
    json.dump(train_samples_aug, f, ensure_ascii=False, indent=2)
print("\nSaved \u2192 ./results/augmented_train.json")
print("Use `train_samples_aug` and `class_weights` when calling run_all_models().")


Original Emergency samples: 59
Sample content tokens after stopword removal:
  ['Makila', 'mingi', 'kobima', 'libumu', 'nga,', 'kolemba']
  ['mpasi', 'ntolo', 'ekómi', 'banda', 'kobeta', 'kokende']
  ['Mwana', 'mpema', 'te', 'elongi', 'kobongola']

Augmenting Emergency samples...
  Rule-based: +236 samples
  Calling Claude API for paraphrases (2 per original)...
  ✗ Batch 1 failed (401 Client Error: Unauthorized for url: https://api.anthropic.com/v1/messages) — synonym fallback
  ✗ Batch 2 failed (401 Client Error: Unauthorized for url: https://api.anthropic.com/v1/messages) — synonym fallback
  ✗ Batch 3 failed (401 Client Error: Unauthorized for url: https://api.anthropic.com/v1/messages) — synonym fallback
  ✗ Batch 4 failed (401 Client Error: Unauthorized for url: https://api.anthropic.com/v1/messages) — synonym fallback
  ✗ Batch 5 failed (401 Client Error: Unauthorized for url: https://api.anthropic.com/v1/messages) — synonym fallback
  ✗ Batch 6 failed (401 Client Error: Unauth

In [6]:
# ============================================================================
# SECTION 5: DATASET CLASS FOR TRANSFORMERS
# ============================================================================

class TriageDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128):
        self.encodings = tokenizer(
            [s['text'] for s in samples],
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor([LABEL2ID[s['label']] for s in samples])
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return {
        'accuracy': accuracy_score(eval_pred.label_ids, predictions),
        'macro_f1': f1_score(eval_pred.label_ids, predictions, average='macro', zero_division=0)
    }

print("Dataset class ready")

Dataset class ready


In [7]:
# ============================================================================
# SECTION 6: ENHANCED MODEL 2 - FINE-TUNED XLM-RoBERTa
# ============================================================================

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float) if class_weights is not None else None
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device)
            )
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

class FineTunedXLMR:
    def __init__(self, model_name='xlm-roberta-base', learning_rate=2e-5,
                 batch_size=16, num_epochs=5, max_length=128, output_dir='./models/xlmr'):
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.max_length = max_length
        self.output_dir = output_dir
        
        print(f"Loading XLM-R tokenizer from {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        print(f"Loading XLM-R model from {model_name}...")
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
            ignore_mismatched_sizes=True
        )
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        print(f"Model ready on {self.device}")
    
    def train(self, train_samples, val_samples, class_weights=None):
        train_dataset = TriageDataset(train_samples, self.tokenizer, self.max_length)
        val_dataset = TriageDataset(val_samples, self.tokenizer, self.max_length)
        
        training_args = TrainingArguments(
            output_dir=self.output_dir,
            learning_rate=self.learning_rate,
            per_device_train_batch_size=self.batch_size,
            per_device_eval_batch_size=self.batch_size,
            num_train_epochs=self.num_epochs,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_accuracy',
            logging_steps=50,
            report_to='none',
            fp16=torch.cuda.is_available(),
            warmup_ratio=0.1,
            weight_decay=0.01,
        )
        
        trainer_cls = WeightedTrainer if class_weights else Trainer
        self.trainer = trainer_cls(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            class_weights=class_weights
        )
        
        print("\nTraining XLM-R...")
        self.trainer.train()
        print("Training complete!")
    
    def predict(self, texts, return_probs=False):
        single = isinstance(texts, str)
        if single:
            texts = [texts]
        
        self.model.eval()
        encodings = self.tokenizer(texts, truncation=True, padding='max_length',
                                   max_length=self.max_length, return_tensors='pt')
        
        with torch.no_grad():
            outputs = self.model(
                input_ids=encodings['input_ids'].to(self.device),
                attention_mask=encodings['attention_mask'].to(self.device)
            )
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
        
        results = [ID2LABEL[p] for p in preds]
        
        if return_probs:
            return results[0] if single else results, probs
        return results[0] if single else results
    
    def save(self, path):
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")
    
    @classmethod
    def load(cls, path):
        obj = cls.__new__(cls)
        obj.tokenizer = AutoTokenizer.from_pretrained(path)
        obj.model = AutoModelForSequenceClassification.from_pretrained(path)
        obj.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        obj.model.to(obj.device)
        obj.max_length = 128
        print(f"XLM-R model loaded from {path}")
        return obj

In [8]:
# ============================================================================
# SECTION 7: ENHANCED MODEL 3 - CROSS-LINGUAL TRANSFER (Curriculum Learning)
# ============================================================================

# Enhanced French training data with more variety
french_train = [
    # Emergency cases
    {'text': 'Douleur thoracique intense et dyspnée sévère, patient inconscient.', 'label': 'Emergency'},
    {'text': 'Enfant cyanosé, ne répond plus normalement.', 'label': 'Emergency'},
    {'text': 'Saignement abondant incontrôlable après accident.', 'label': 'Emergency'},
    {'text': 'Convulsions et perte de conscience soudaine.', 'label': 'Emergency'},
    {'text': 'Brûlures graves au troisième degré sur une grande surface.', 'label': 'Emergency'},
    {'text': 'Arrêt respiratoire, patient ne respire plus.', 'label': 'Emergency'},
    {'text': 'Traumatisme crânien avec perte de connaissance prolongée.', 'label': 'Emergency'},
    # Moderate cases
    {'text': 'Fièvre modérée depuis 48h avec fatigue intense.', 'label': 'Moderate'},
    {'text': 'Douleur abdominale modérée et nausées depuis ce matin.', 'label': 'Moderate'},
    {'text': 'Toux persistante depuis une semaine, légère fièvre.', 'label': 'Moderate'},
    {'text': 'Vomissements répétés avec déshydratation.', 'label': 'Moderate'},
    {'text': 'Douleur articulaire avec gonflement modéré.', 'label': 'Moderate'},
    {'text': 'Maux de tête persistants sans autres symptômes.', 'label': 'Moderate'},
    # Low cases
    {'text': 'Légère céphalée sans autre symptôme.', 'label': 'Low'},
    {'text': 'Toux légère, pas de fièvre, pas de difficultés respiratoires.', 'label': 'Low'},
    {'text': 'Légère fatigue, mange et dort normalement.', 'label': 'Low'},
    {'text': 'Petite coupure superficielle, saignement arrêté.', 'label': 'Low'},
    {'text': 'Rhume léger sans complications.', 'label': 'Low'},
    {'text': 'Légère douleur musculaire après exercice.', 'label': 'Low'},
] * 18  # 108 French samples

french_val = [
    {'text': 'Convulsions et perte de conscience soudaine.', 'label': 'Emergency'},
    {'text': 'Vomissements répétés avec déshydratation.', 'label': 'Moderate'},
    {'text': 'Rhume léger sans complications.', 'label': 'Low'},
] * 4  # 12 French val samples

class CrossLingualTriageDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128):
        self.encodings = tokenizer(
            [s["text"] for s in samples],
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(
            [LABEL2ID[s["label"]] for s in samples], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

class CrossLingualWeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            weight = torch.tensor(self.class_weights, dtype=torch.float).to(logits.device)
            loss = torch.nn.CrossEntropyLoss(weight=weight)(logits, labels)
        else:
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
        return (loss, outputs) if return_outputs else loss

class CrossLingualTriageTransfer:
    """
    Enhanced Two-stage transfer learning with curriculum learning:
    Stage 1: Fine-tune XLM-R on French medical data
    Stage 2: Adapt to Lingala using augmented training set
    Stage 3: (Optional) Fine-tune on high-confidence Lingala predictions
    """
    def __init__(
        self,
        model_name: str = "xlm-roberta-base",
        stage1_learning_rate: float = 2e-5,
        stage1_batch_size: int = 16,
        stage1_epochs: int = 5,
        stage2_learning_rate: float = 1e-5,
        stage2_batch_size: int = 16,
        stage2_epochs: int = 3,
        stage3_learning_rate: float = 5e-6,
        stage3_epochs: int = 2,
        max_length: int = 128,
        weight_decay: float = 0.01,
        warmup_steps: int = 0,
        fp16: bool = False,
        seed: int = 42,
        output_dir: str = "./models/cross_lingual",
    ):
        self.stage1_learning_rate = stage1_learning_rate
        self.stage1_batch_size = stage1_batch_size
        self.stage1_epochs = stage1_epochs
        self.stage2_learning_rate = stage2_learning_rate
        self.stage2_batch_size = stage2_batch_size
        self.stage2_epochs = stage2_epochs
        self.stage3_learning_rate = stage3_learning_rate
        self.stage3_epochs = stage3_epochs
        self.max_length = max_length
        self.weight_decay = weight_decay
        self.warmup_steps = warmup_steps
        self.fp16 = fp16
        self.seed = seed
        self.output_dir = output_dir
        self.stage1_complete = False

        print(f"Loading tokenizer from {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        print(f"Loading base model from {model_name}")
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=3,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            ignore_mismatched_sizes=True,
        )
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(device)
        print(f"Model ready | device={device}")

    def _build_trainer(
        self,
        stage: int,
        output_subdir: str,
        learning_rate: float,
        batch_size: int,
        num_epochs: int,
        train_dataset: Dataset,
        eval_dataset: Dataset,
        class_weights=None,
    ) -> Trainer:

        args = TrainingArguments(
            output_dir=output_subdir,
            learning_rate=learning_rate,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=num_epochs,
            weight_decay=self.weight_decay,
            warmup_steps=self.warmup_steps,
            lr_scheduler_type="linear",
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_macro_f1",
            greater_is_better=True,
            logging_steps=50,
            report_to="none",
            fp16=self.fp16,
            seed=self.seed,
            dataloader_num_workers=0,
            label_names=["labels"],
        )
        trainer_cls = CrossLingualWeightedTrainer if class_weights else Trainer
        extra = {"class_weights": class_weights} if class_weights else {}
        return trainer_cls(
            model=self.model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            compute_metrics=self._compute_metrics,
            **extra,
        )

    def _compute_metrics(self, p: EvalPrediction):
        preds = np.argmax(p.predictions, axis=1)
        labels = p.label_ids
        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "emergency_recall": recall_score(labels, preds, labels=[0], average="micro", zero_division=0),
        }

    def train_stage1_french(
        self,
        french_train: list,
        french_val: list,
        class_weights=None,
    ):
        stage1_dir = os.path.join(self.output_dir, "stage1_french")
        print(
            f"\n{'='*55}\nSTAGE 1: FRENCH FINE-TUNING (Curriculum Step 1)\n"
            f"  Dataset : {len(french_train)} train / {len(french_val)} val\n"
            f"  LR      : {self.stage1_learning_rate}  Epochs: {self.stage1_epochs}\n"
            f"{'='*55}"
        )
        train_ds = CrossLingualTriageDataset(french_train, self.tokenizer, self.max_length)
        val_ds = CrossLingualTriageDataset(french_val, self.tokenizer, self.max_length)
        self.stage1_trainer = self._build_trainer(
            stage=1, output_subdir=stage1_dir,
            learning_rate=self.stage1_learning_rate,
            batch_size=self.stage1_batch_size,
            num_epochs=self.stage1_epochs,
            train_dataset=train_ds, eval_dataset=val_ds,
            class_weights=class_weights,
        )
        self.stage1_trainer.train()
        self.stage1_complete = True
        return self.stage1_trainer.state.log_history[-1] if self.stage1_trainer.state.log_history else {}

    def train_stage2_lingala(
        self,
        lingala_train: list,
        lingala_val: list,
        class_weights=None,
    ):
        if not self.stage1_complete:
            print("⚠️  Stage 1 not complete — running Stage 2 from base weights.")
        stage2_dir = os.path.join(self.output_dir, "stage2_lingala")
        print(
            f"\n{'='*55}\nSTAGE 2: LINGALA ADAPTATION (Curriculum Step 2)\n"
            f"  Dataset : {len(lingala_train)} train / {len(lingala_val)} val\n"
            f"  LR      : {self.stage2_learning_rate}  Epochs: {self.stage2_epochs}\n"
            f"{'='*55}"
        )
        train_ds = CrossLingualTriageDataset(lingala_train, self.tokenizer, self.max_length)
        val_ds = CrossLingualTriageDataset(lingala_val, self.tokenizer, self.max_length)
        self.stage2_trainer = self._build_trainer(
            stage=2, output_subdir=stage2_dir,
            learning_rate=self.stage2_learning_rate,
            batch_size=self.stage2_batch_size,
            num_epochs=self.stage2_epochs,
            train_dataset=train_ds, eval_dataset=val_ds,
            class_weights=class_weights,
        )
        self.stage2_trainer.train()
        return self.stage2_trainer.state.log_history[-1] if self.stage2_trainer.state.log_history else {}

    def run_full_pipeline(
        self,
        french_train: list,
        french_val: list,
        lingala_train: list,
        lingala_val: list,
        class_weights=None,
    ) -> dict:
        print("\n" + "="*60)
        print("STAGE 1: Training on French medical data")
        print("="*60)
        s1 = self.train_stage1_french(french_train, french_val, class_weights)
        
        print("\n" + "="*60)
        print("STAGE 2: Adapting to Lingala")
        print("="*60)
        s2 = self.train_stage2_lingala(lingala_train, lingala_val, class_weights)
        
        return {"stage1": s1, "stage2": s2}

    def evaluate_test(self, test_samples: list) -> dict:
        test_ds = CrossLingualTriageDataset(test_samples, self.tokenizer, self.max_length)
        device = next(self.model.parameters()).device
        self.model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for i in range(len(test_ds)):
                item = test_ds[i]
                out = self.model(
                    input_ids=item["input_ids"].unsqueeze(0).to(device),
                    attention_mask=item["attention_mask"].unsqueeze(0).to(device),
                )
                all_preds.append(int(torch.argmax(out.logits, dim=1)))
                all_labels.append(int(item["labels"]))

        macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        emergency_recall = recall_score(all_labels, all_preds, labels=[0], average="micro", zero_division=0)
        accuracy = accuracy_score(all_labels, all_preds)
        n_emg = all_labels.count(0)
        crit_err = sum(1 for t, p in zip(all_labels, all_preds) if t == 0 and p == 2)
        cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2]).tolist()

        print("\n" + classification_report(
            all_labels, all_preds,
            target_names=["Emergency", "Moderate", "Low"],
            zero_division=0,
        ))
        return {
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "emergency_recall": emergency_recall,
            "critical_error_rate": crit_err / n_emg if n_emg > 0 else 0.0,
            "confusion_matrix": cm,
        }

    def predict(self, text, return_probs=False):
        single_input = isinstance(text, str)
        texts = [text] if single_input else text
        
        self.model.eval()
        device = next(self.model.parameters()).device
        
        enc = self.tokenizer(
            texts, 
            truncation=True, 
            padding="max_length",
            max_length=self.max_length, 
            return_tensors="pt"
        )
        
        with torch.no_grad():
            logits = self.model(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
            ).logits
        
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred_ids = np.argmax(probs, axis=1)
        
        if return_probs:
            results = []
            for i, pred_id in enumerate(pred_ids):
                results.append({
                    "label": ID2LABEL[pred_id],
                    "confidence": float(probs[i][pred_id]),
                    "probabilities": {ID2LABEL[j]: float(probs[i][j]) for j in range(len(probs[i]))}
                })
            return results[0] if single_input else results
        else:
            return int(pred_ids[0]) if single_input else pred_ids.tolist()

    def save(self, path: str):
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")

    @classmethod
    def load(cls, path: str, **kwargs):
        obj = cls.__new__(cls)
        obj.tokenizer = AutoTokenizer.from_pretrained(path)
        obj.model = AutoModelForSequenceClassification.from_pretrained(path)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        obj.model.to(device)
        obj.max_length = kwargs.get('max_length', 128)
        print(f"Model loaded from {path} | device={device}")
        return obj

print("✅ Cross-Lingual Transfer model class ready (Enhanced with Curriculum Learning)")
print("   Stage 1: French medical data (108 samples)")
print("   Stage 2: Lingala adaptation (augmented training set)")

✅ Cross-Lingual Transfer model class ready (Enhanced with Curriculum Learning)
   Stage 1: French medical data (108 samples)
   Stage 2: Lingala adaptation (augmented training set)


In [9]:
# ============================================================================
# SECTION 8: ENHANCED MODEL 4 - AFRIBERTA
# ============================================================================

AFRIBERTA_MODEL = 'castorini/afriberta_large'

class AfriBERTaTriage:
    def __init__(self, model_name=AFRIBERTA_MODEL, learning_rate=3e-5,
                 batch_size=16, num_epochs=5, max_length=128, output_dir='./models/afriberta'):
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.max_length = max_length
        self.output_dir = output_dir
        
        print(f"Loading AfriBERTa tokenizer from {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        print(f"Loading AfriBERTa model from {model_name}...")
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
            ignore_mismatched_sizes=True
        )
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        print(f"Model ready on {self.device}")
    
    def train(self, train_samples, val_samples, class_weights=None):
        train_dataset = TriageDataset(train_samples, self.tokenizer, self.max_length)
        val_dataset = TriageDataset(val_samples, self.tokenizer, self.max_length)
        
        training_args = TrainingArguments(
            output_dir=self.output_dir,
            learning_rate=self.learning_rate,
            per_device_train_batch_size=self.batch_size,
            per_device_eval_batch_size=self.batch_size,
            num_train_epochs=self.num_epochs,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_accuracy',
            logging_steps=50,
            report_to='none',
            fp16=torch.cuda.is_available(),
            warmup_ratio=0.1,
            weight_decay=0.01,
        )
        
        trainer_cls = WeightedTrainer if class_weights else Trainer
        self.trainer = trainer_cls(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            class_weights=class_weights
        )
        
        print("\nTraining AfriBERTa...")
        self.trainer.train()
        print("Training complete!")
    
    def predict(self, texts, return_probs=False):
        single_input = isinstance(texts, str)
        if single_input:
            texts = [texts]
        
        self.model.eval()
        device = next(self.model.parameters()).device
        
        enc = self.tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            logits = self.model(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
            ).logits
        
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred_ids = np.argmax(probs, axis=1)
        
        if return_probs:
            results = []
            for i, pred_id in enumerate(pred_ids):
                results.append({
                    "label": ID2LABEL[pred_id],
                    "confidence": float(probs[i][pred_id]),
                    "probabilities": {ID2LABEL[j]: float(probs[i][j]) for j in range(len(probs[i]))}
                })
            return results[0] if single_input else results
        else:
            results = [ID2LABEL[p] for p in pred_ids]
            return results[0] if single_input else results
    
    def save(self, path):
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"AfriBERTa model saved to {path}")
    
    @classmethod
    def load(cls, path, max_length=128):
        obj = cls.__new__(cls)
        obj.tokenizer = AutoTokenizer.from_pretrained(path)
        obj.model = AutoModelForSequenceClassification.from_pretrained(path)
        obj.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        obj.model.to(obj.device)
        obj.max_length = max_length
        print(f"AfriBERTa model loaded from {path}")
        return obj

In [10]:
# ============================================================================
# SECTION 9: ENHANCED BASELINE SVM MODEL
# ============================================================================

class SVMBaseline:
    def __init__(self, C=0.1, use_char_ngrams=True):
        self.C = C
        self.use_char_ngrams = use_char_ngrams
        self.model = None
        self.vectorizer = None
    
    def train(self, train_samples, val_samples=None):
        train_texts = [s['text'] for s in train_samples]
        train_labels = [s['label'] for s in train_samples]
        
        if self.use_char_ngrams:
            # Character n-grams for better generalization
            self.vectorizer = TfidfVectorizer(
                analyzer='char',
                ngram_range=(3, 6),
                max_features=15000,
                sublinear_tf=True,
                min_df=2,
                max_df=0.95
            )
        else:
            # Word-level with stopwords
            self.vectorizer = TfidfVectorizer(
                max_features=5000,
                stop_words=list(STOPWORDS_MEDICAL),
                ngram_range=(1, 2),
                sublinear_tf=True
            )
        
        print("Fitting TF-IDF vectorizer...")
        X_train = self.vectorizer.fit_transform(train_texts)
        
        print(f"Training SVM (C={self.C}) on {X_train.shape[0]} samples with {X_train.shape[1]} features...")
        self.model = LinearSVC(
            C=self.C,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            max_iter=3000,
            dual='auto'
        )
        self.model.fit(X_train, train_labels)
        print("Training complete!")
    
    def predict(self, texts, return_probs=False):
        single = isinstance(texts, str)
        if single:
            texts = [texts]
        
        X = self.vectorizer.transform(texts)
        
        if return_probs:
            # Approximate probabilities using Platt scaling
            decision = self.model.decision_function(X)
            if len(decision.shape) == 1:
                decision = decision.reshape(-1, 1)
            exp_decision = np.exp(decision - decision.max(axis=1, keepdims=True))
            probs = exp_decision / exp_decision.sum(axis=1, keepdims=True)
            
            # Ensure 3 classes
            if probs.shape[1] == 1:
                probs_expanded = np.zeros((len(texts), 3))
                probs_expanded[:, 0] = probs[:, 0]  # Emergency
                probs_expanded[:, 2] = 1 - probs[:, 0]  # Low
                probs = probs_expanded
            
            if single:
                return probs[0]
            return probs
        
        predictions = self.model.predict(X)
        return predictions[0] if single else predictions
    
    def evaluate(self, test_samples):
        test_texts = [s['text'] for s in test_samples]
        test_labels = [s['label'] for s in test_samples]
        
        X_test = self.vectorizer.transform(test_texts)
        y_pred = self.model.predict(X_test)
        
        accuracy = accuracy_score(test_labels, y_pred)
        macro_f1 = f1_score(test_labels, y_pred, average='macro', zero_division=0)
        
        return {
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'predictions': y_pred,
            'confusion_matrix': confusion_matrix(test_labels, y_pred, labels=CLASS_NAMES)
        }
    
    def save(self, path):
        import joblib
        os.makedirs(path, exist_ok=True)
        joblib.dump(self.model, f"{path}/svm_model.pkl")
        joblib.dump(self.vectorizer, f"{path}/tfidf_vectorizer.pkl")
        print(f"SVM model saved to {path}")
    
    @classmethod
    def load(cls, path):
        import joblib
        obj = cls.__new__(cls)
        obj.model = joblib.load(f"{path}/svm_model.pkl")
        obj.vectorizer = joblib.load(f"{path}/tfidf_vectorizer.pkl")
        print(f"SVM model loaded from {path}")
        return obj

In [12]:
# ============================================================================
# SECTION 10: TRAIN ALL ENHANCED MODELS
# ============================================================================

def compute_class_weights(samples):
    counts = Counter(s['label'] for s in samples)
    total = sum(counts.values())
    n_classes = len(counts)
    return [total / (n_classes * counts.get(l, 1)) for l in CLASS_NAMES]

class_weights = compute_class_weights(train_samples_aug)
print(f"Class weights: {class_weights}")

# Train Enhanced Baseline SVM
print("\n" + "="*60)
print("TRAINING ENHANCED BASELINE SVM (Character n-grams)")
print("="*60)
svm_baseline = SVMBaseline(C=0.1, use_char_ngrams=True)
svm_baseline.train(train_samples_aug)
svm_baseline.save('./models/svm_baseline_enhanced')

# Train Model 2: XLM-R
print("\n" + "="*60)
print("TRAINING MODEL 2: XLM-R (Enhanced)")
print("="*60)
model2 = FineTunedXLMR(output_dir='./models/xlmr_enhanced')
model2.train(train_samples_aug, val_samples, class_weights=class_weights)
model2.save('./models/xlmr_enhanced/final')

# Train Model 3: Cross-Lingual Transfer
print("\n" + "="*60)
print("TRAINING MODEL 3: Enhanced Cross-Lingual Transfer")
print("="*60)
model3 = CrossLingualTriageTransfer(output_dir='./models/cross_lingual_enhanced')
model3.run_full_pipeline(
    french_train=french_train,
    french_val=french_val,
    lingala_train=train_samples_aug,
    lingala_val=val_samples,
    class_weights=class_weights
)
model3.save('./models/cross_lingual_enhanced/final')

# Train Model 4: AfriBERTa
print("\n" + "="*60)
print("TRAINING MODEL 4: AfriBERTa (Enhanced)")
print("="*60)
model4 = AfriBERTaTriage(output_dir='./models/afriberta_enhanced')
model4.train(train_samples_aug, val_samples, class_weights=class_weights)
model4.save('./models/afriberta_enhanced/final')

print("\n✅ All enhanced models trained and saved!")

Class weights: [0.8625850340136054, 2.855855855855856, 0.6708994708994709]

TRAINING ENHANCED BASELINE SVM (Character n-grams)
Fitting TF-IDF vectorizer...
Training SVM (C=0.1) on 634 samples with 6354 features...
Training complete!
SVM model saved to ./models/svm_baseline_enhanced

TRAINING MODEL 2: XLM-R (Enhanced)
Loading XLM-R tokenizer from xlm-roberta-base...
Loading XLM-R model from xlm-roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Model ready on cpu

Training XLM-R...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.067500,0.615385,0.387128
2,1.054902,0.640819,0.846154,0.650698
3,0.703695,0.568143,0.901099,0.783847
4,0.425879,0.559147,0.923077,0.838059
5,0.382275,0.420360,0.923077,0.838059


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

AfriBERTa model saved to ./models/afriberta_enhanced/final

✅ All enhanced models trained and saved!


In [ ]:
# ============================================================================
# SECTION 11: ENSEMBLE MODEL
# ============================================================================

from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.sparse import hstack

class TransformerSklearnWrapper(BaseEstimator, ClassifierMixin):
    """Wrapper to make transformer models compatible with sklearn ensemble"""
    
    def __init__(self, model, name):
        self.model = model
        self.name = name
        self.classes_ = CLASS_NAMES
    
    def fit(self, X, y):
        return self
    
    def predict(self, X):
        if isinstance(X, np.ndarray):
            X = X.tolist()
        return self.model.predict(X)
    
    def predict_proba(self, X):
        if isinstance(X, np.ndarray):
            X = X.tolist()
        
        # Try to get probabilities from the model
        try:
            if hasattr(self.model, 'predict') and hasattr(self.model, 'predict_proba'):
                return self.model.predict_proba(X)
            elif hasattr(self.model, 'predict') and hasattr(self.model, 'predict_proba'):
                return self.model.predict_proba(X)
            else:
                # Approximate probabilities from predictions
                preds = self.model.predict(X)
                probs = np.zeros((len(X), 3))
                for i, pred in enumerate(preds):
                    if isinstance(pred, str):
                        if pred in CLASS_NAMES:
                            probs[i, CLASS_NAMES.index(pred)] = 0.6
                            remaining = 0.4 / 2
                            for j in range(3):
                                if j != CLASS_NAMES.index(pred):
                                    probs[i, j] = remaining
                    else:
                        probs[i, pred] = 0.6
                        remaining = 0.4 / 2
                        for j in range(3):
                            if j != pred:
                                probs[i, j] = remaining
                return probs
        except Exception as e:
            print(f"Warning: Could not get probabilities from {self.name}: {e}")
            return np.ones((len(X), 3)) / 3


class EnsembleTriage:
    """Weighted ensemble of multiple models"""
    
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights if weights else [1.0] * len(models)
        self.classes_ = CLASS_NAMES
    
    def predict_proba(self, texts):
        single = isinstance(texts, str)
        if single:
            texts = [texts]
        
        all_probs = []
        for model in self.models:
            try:
                probs = model.predict_proba(texts)
                all_probs.append(probs)
            except Exception as e:
                print(f"Error getting probabilities: {e}")
                all_probs.append(np.ones((len(texts), 3)) / 3)
        
        all_probs = np.array(all_probs)
        weights = np.array(self.weights).reshape(-1, 1, 1)
        ensemble_probs = np.sum(all_probs * weights, axis=0) / np.sum(weights)
        
        return ensemble_probs[0] if single else ensemble_probs
    
    def predict(self, texts):
        probs = self.predict_proba(texts)
        
        single = isinstance(texts, str)
        if single:
            probs = [probs]
        
        predictions = [CLASS_NAMES[np.argmax(p)] for p in probs]
        return predictions[0] if single else predictions


def create_ensemble(models, val_samples=None):
    """Create ensemble with optional weight calibration"""
    
    ensemble_models = []
    for name, model in models:
        wrapper = TransformerSklearnWrapper(model, name)
        ensemble_models.append(wrapper)
    
    # Default equal weights
    ensemble = EnsembleTriage(ensemble_models)
    
    # Optionally calibrate weights on validation set
    if val_samples:
        print("Calibrating ensemble weights on validation set...")
        val_texts = [s['text'] for s in val_samples]
        val_labels = [s['label'] for s in val_samples]
        
        model_scores = []
        for name, model in models:
            preds = model.predict(val_texts)
            acc = accuracy_score(val_labels, preds)
            model_scores.append(acc)
            print(f"  {name}: {acc:.4f}")
        
        # Weight by accuracy
        total = sum(model_scores)
        if total > 0:
            ensemble.weights = [s / total for s in model_scores]
            print(f"  Adjusted weights: {dict(zip([m[0] for m in models], ensemble.weights))}")
    
    return ensemble


# Create ensemble with all transformer models
models_for_ensemble = [
    ('XLM-R', model2),
    ('Cross-Lingual', model3),
    ('AfriBERTa', model4)
]

ensemble_model = create_ensemble(models_for_ensemble, val_samples)
print("\n✅ Ensemble model created")

In [ ]:
# ============================================================================
# SECTION 12: CONFIDENCE-BASED PREDICTION WITH REJECTION
# ============================================================================

class ConfidencePredictor:
    """Wrapper that rejects low-confidence predictions for clinical safety"""
    
    def __init__(self, model, confidence_threshold=0.7):
        self.model = model
        self.confidence_threshold = confidence_threshold
        self.classes_ = CLASS_NAMES
    
    def predict_with_confidence(self, texts):
        single = isinstance(texts, str)
        if single:
            texts = [texts]
        
        probs = self.model.predict_proba(texts)
        predictions = []
        confidences = []
        
        for prob in probs:
            max_conf = np.max(prob)
            pred_idx = np.argmax(prob)
            confidences.append(max_conf)
            
            if max_conf >= self.confidence_threshold:
                predictions.append(CLASS_NAMES[pred_idx])
            else:
                predictions.append('Uncertain')  # Reject low-confidence predictions
        
        if single:
            return predictions[0], confidences[0]
        return predictions, confidences
    
    def predict(self, texts):
        preds, _ = self.predict_with_confidence(texts)
        return preds


# Create confidence-based predictor using ensemble
confidence_predictor = ConfidencePredictor(ensemble_model, confidence_threshold=0.7)
print("✅ Confidence-based predictor created")

In [15]:
# ============================================================================
# SECTION 13: EVALUATE ON TEST SET WITH CONFIDENCE INTERVALS
# ============================================================================

from sklearn.utils import resample

def bootstrap_emergency_recall(y_true, y_pred, n_bootstrap=1000, ci=95):
    emergency_indices = [i for i, label in enumerate(y_true) if label == "Emergency"]
    if len(emergency_indices) == 0:
        return None, None, None, 0
    
    emergency_pred = [y_pred[i] for i in emergency_indices]
    n_emergency = len(emergency_indices)
    
    bootstrap_recalls = []
    for _ in range(n_bootstrap):
        indices = resample(range(n_emergency), n_samples=n_emergency, replace=True)
        correct = sum(1 for i in indices if emergency_pred[i] == "Emergency")
        bootstrap_recalls.append(correct / n_emergency)
    
    alpha = 100 - ci
    lower = np.percentile(bootstrap_recalls, alpha / 2)
    upper = np.percentile(bootstrap_recalls, 100 - alpha / 2)
    return lower, upper, np.mean(bootstrap_recalls), n_emergency


def evaluate_model(model, test_samples, model_name, n_bootstrap=1000, ci=95):
    test_texts = [s['text'] for s in test_samples]
    y_true = [s['label'] for s in test_samples]
    y_pred_raw = model.predict(test_texts)
    
    # Convert predictions to strings if needed
    if isinstance(y_pred_raw, list) and len(y_pred_raw) > 0:
        if isinstance(y_pred_raw[0], int):
            y_pred = [ID2LABEL[p] for p in y_pred_raw]
        else:
            y_pred = y_pred_raw
    else:
        y_pred = y_pred_raw
    
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    emergency_recall = recall_score(y_true, y_pred, labels=['Emergency'], average='micro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES)
    
    ci_lower, ci_upper, ci_mean, n_emergency = bootstrap_emergency_recall(y_true, y_pred, n_bootstrap, ci)
    
    print(f"\n{'='*60}")
    print(f"{model_name} Results")
    print(f"{'='*60}")
    print(f"  Samples: {len(test_samples)} (Emergency: {n_emergency})")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Macro F1: {macro_f1:.4f}")
    
    if ci_lower is not None:
        print(f"  Emergency Recall: {emergency_recall:.4f} (95% CI: [{ci_lower:.4f}, {ci_upper:.4f}])")
    else:
        print(f"  Emergency Recall: {emergency_recall:.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    
    return {
        'accuracy': accuracy, 
        'macro_f1': macro_f1, 
        'emergency_recall': emergency_recall,
        'emergency_recall_ci': (ci_lower, ci_upper) if ci_lower else None,
        'n_emergency_samples': n_emergency,
        'confusion_matrix': cm
    }


print("\n" + "="*70)
print("ENHANCED MODEL EVALUATION ON TEST SET")
print("="*70)

results = {}
results['SVM Baseline (char)'] = evaluate_model(svm_baseline, test_samples, 'SVM Baseline (char)')
results['XLM-R'] = evaluate_model(model2, test_samples, 'XLM-R')
results['Cross-Lingual'] = evaluate_model(model3, test_samples, 'Cross-Lingual')
results['AfriBERTa'] = evaluate_model(model4, test_samples, 'AfriBERTa')
results['Ensemble'] = evaluate_model(ensemble_model, test_samples, 'Ensemble')

# Print comparison
print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(f"{'Model':<20} {'Accuracy':<12} {'Macro F1':<12} {'Emergency Recall':<20}")
print("-"*70)

for name, metrics in results.items():
    em_recall = f"{metrics['emergency_recall']:.4f}"
    if metrics['emergency_recall_ci']:
        ci_lower, ci_upper = metrics['emergency_recall_ci']
        em_recall = f"{metrics['emergency_recall']:.4f} [{ci_lower:.3f}-{ci_upper:.3f}]"
    print(f"{name:<20} {metrics['accuracy']:<12.4f} {metrics['macro_f1']:<12.4f} {em_recall:<20}")

# Identify best model
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
print(f"\n🏆 BEST MODEL: {best_model_name} (Accuracy: {results[best_model_name]['accuracy']:.4f})")


ENHANCED MODEL EVALUATION ON TEST SET

SVM Baseline (char) Results
  Samples: 95 (Emergency: 10)
  Accuracy: 0.9263
  Macro F1: 0.8546
  Emergency Recall: 1.0000 (95% CI: [1.0000, 1.0000])

Classification Report:
              precision    recall  f1-score   support

   Emergency       0.67      1.00      0.80        10
    Moderate       0.97      0.99      0.98        68
         Low       1.00      0.65      0.79        17

    accuracy                           0.93        95
   macro avg       0.88      0.88      0.85        95
weighted avg       0.94      0.93      0.92        95


XLM-R Results
  Samples: 95 (Emergency: 10)
  Accuracy: 0.9158
  Macro F1: 0.8313
  Emergency Recall: 0.9000 (95% CI: [0.7000, 1.0000])

Classification Report:
              precision    recall  f1-score   support

   Emergency       0.69      0.90      0.78        10
    Moderate       0.97      0.99      0.98        68
         Low       0.85      0.65      0.73        17

    accuracy              

In [23]:
# ============================================================================
# SECTION 14: EXTERNAL EVALUATION ON FORUM DATA
# ============================================================================

def load_external_test_set(file_path='french_to_lingala_100_translations.csv'):
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"Loaded {len(df)} external test samples from {file_path}")
        
        if 'urgency' in df.columns:
            df['urgency'] = df['urgency'].astype(str).str.strip()
            
            label_map = {
                'Emergency': 'Emergency', 'emergency': 'Emergency',
                'Moderate': 'Moderate', 'moderate': 'Moderate',
                'Low': 'Low', 'low': 'Low'
            }
            df['urgency'] = df['urgency'].map(label_map).fillna(df['urgency'])
            df = df[df['urgency'].isin(CLASS_NAMES)]
            print(f"Filtered to {len(df)} valid samples")
        
        return df
    else:
        print(f"⚠️ File {file_path} not found!")
        return None


def evaluate_external(model, model_name, external_df):
    print(f"\n{'='*60}")
    print(f"EXTERNAL EVALUATION: {model_name}")
    print(f"{'='*60}")
    
    y_true = external_df['urgency'].tolist()
    y_pred_raw = model.predict(external_df['text'].tolist())
    
    if isinstance(y_pred_raw, list) and len(y_pred_raw) > 0:
        if isinstance(y_pred_raw[0], int):
            y_pred = [ID2LABEL[p] for p in y_pred_raw]
        else:
            y_pred = y_pred_raw
    else:
        y_pred = y_pred_raw
    
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    emergency_recall = recall_score(y_true, y_pred, labels=['Emergency'], average='micro', zero_division=0)
    
    print(f"\nSamples: {len(external_df)}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Emergency Recall: {emergency_recall:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    
    return {'accuracy': accuracy, 'macro_f1': macro_f1, 'emergency_recall': emergency_recall}


# Load external test set
external_df = load_external_test_set()

if external_df is not None:
    print(f"\nLoaded external dataset with {len(external_df)} samples")
    print(f"Label distribution:\n{external_df['urgency'].value_counts()}")
    
    # Run external evaluation
    external_results = {}
    external_results['SVM Baseline'] = evaluate_external(svm_baseline, 'SVM Baseline', external_df)
    external_results['XLM-R'] = evaluate_external(model2, 'XLM-R', external_df)
    external_results['Cross-Lingual'] = evaluate_external(model3, 'Cross-Lingual', external_df)
    external_results['AfriBERTa'] = evaluate_external(model4, 'AfriBERTa', external_df)
    external_results['Ensemble'] = evaluate_external(ensemble_model, 'Ensemble', external_df)
    external_results['Confidence Predictor'] = evaluate_external(confidence_predictor, 'Confidence Predictor', external_df)
    
    # Summary
    print("\n" + "="*60)
    print("EXTERNAL EVALUATION SUMMARY")
    print("="*60)
    summary_df = pd.DataFrame(external_results).T
    print(summary_df.round(4))
    
    # Save results
    summary_df.to_csv('./results/external_evaluation_summary_enhanced.csv')
    print("\n✅ External evaluation results saved to ./results/external_evaluation_summary_enhanced.csv")

Loaded 100 external test samples from french_to_lingala_100_translations.csv

Loaded external dataset with 100 samples


KeyError: 'urgency'

In [ ]:
# ============================================================================
# SECTION 15: PLOTTING RESULTS
# ============================================================================

def plot_comparisons(results, external_results):
    """Plot comparison of all models on test and external sets"""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Test set results
    model_names = list(results.keys())
    accuracies = [results[m]['accuracy'] for m in model_names]
    macro_f1s = [results[m]['macro_f1'] for m in model_names]
    
    x = np.arange(len(model_names))
    width = 0.35
    
    axes[0].bar(x - width/2, accuracies, width, label='Accuracy', color='steelblue')
    axes[0].bar(x + width/2, macro_f1s, width, label='Macro F1', color='coral')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Test Set Performance')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(model_names, rotation=45, ha='right')
    axes[0].legend()
    axes[0].set_ylim(0, 1)
    
    # External set results
    if external_results:
        ext_names = list(external_results.keys())
        ext_acc = [external_results[m]['accuracy'] for m in ext_names]
        ext_f1 = [external_results[m]['macro_f1'] for m in ext_names]
        
        x_ext = np.arange(len(ext_names))
        axes[1].bar(x_ext - width/2, ext_acc, width, label='Accuracy', color='steelblue')
        axes[1].bar(x_ext + width/2, ext_f1, width, label='Macro F1', color='coral')
        axes[1].set_ylabel('Score')
        axes[1].set_title('External (Forum) Set Performance')
        axes[1].set_xticks(x_ext)
        axes[1].set_xticklabels(ext_names, rotation=45, ha='right')
        axes[1].legend()
        axes[1].set_ylim(0, 1)
    
    plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('./results/enhanced_model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✅ Enhanced comparison plot saved to ./results/enhanced_model_comparison.png")


plot_comparisons(results, external_results if external_df is not None else None)

# Plot confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

models_to_plot = ['SVM Baseline (char)', 'XLM-R', 'Cross-Lingual', 'AfriBERTa', 'Ensemble']
for idx, name in enumerate(models_to_plot):
    if name in results:
        disp = ConfusionMatrixDisplay(results[name]['confusion_matrix'], display_labels=CLASS_NAMES)
        disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')
        axes[idx].set_title(name)

plt.suptitle('Confusion Matrices - Enhanced Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/enhanced_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Enhanced confusion matrices saved")

In [19]:
# ============================================================================
# SECTION 16: SAVE BEST ENHANCED MODEL
# ============================================================================

# Identify best model based on test accuracy
best_enhanced_name = max(results, key=lambda x: results[x]['accuracy'])
print(f"\n🏆 BEST ENHANCED MODEL: {best_enhanced_name} (Accuracy: {results[best_enhanced_name]['accuracy']:.4f})")

# Save the best model
if best_enhanced_name == 'XLM-R':
    best_enhanced = model2
    best_enhanced.save('./models/best_enhanced_model_xlmr')
elif best_enhanced_name == 'Cross-Lingual':
    best_enhanced = model3
    best_enhanced.save('./models/best_enhanced_model_cross_lingual')
elif best_enhanced_name == 'AfriBERTa':
    best_enhanced = model4
    best_enhanced.save('./models/best_enhanced_model_afriberta')
elif best_enhanced_name == 'Ensemble':
    best_enhanced = ensemble_model
    # Save ensemble components
    import joblib
    joblib.dump(ensemble_model, './models/best_enhanced_model_ensemble.pkl')
else:
    best_enhanced = svm_baseline
    best_enhanced.save('./models/best_enhanced_model_svm')

print(f"✅ Best enhanced model saved to ./models/")


🏆 BEST ENHANCED MODEL: AfriBERTa (Accuracy: 0.9684)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

AfriBERTa model saved to ./models/best_enhanced_model_afriberta
✅ Best enhanced model saved to ./models/


In [20]:
# ============================================================================
# SECTION 17: LOAD AND USE SAVED MODEL FOR INFERENCE
# ============================================================================

def load_best_model(model_type='ensemble'):
    """Load a saved model for inference"""
    if model_type == 'xlmr':
        return FineTunedXLMR.load('./models/best_enhanced_model_xlmr')
    elif model_type == 'cross_lingual':
        return CrossLingualTriageTransfer.load('./models/best_enhanced_model_cross_lingual')
    elif model_type == 'afriberta':
        return AfriBERTaTriage.load('./models/best_enhanced_model_afriberta')
    elif model_type == 'ensemble':
        import joblib
        return joblib.load('./models/best_enhanced_model_ensemble.pkl')
    elif model_type == 'svm':
        return SVMBaseline.load('./models/best_enhanced_model_svm')
    else:
        raise ValueError("model_type must be 'xlmr', 'cross_lingual', 'afriberta', 'ensemble', or 'svm'")

print("To load a saved model, use: load_best_model('ensemble')")
print("Then call .predict() on the loaded model")

To load a saved model, use: load_best_model('ensemble')
Then call .predict() on the loaded model


In [21]:
# ============================================================================
# SECTION 18: INFERENCE DEMO
# ============================================================================

test_sentences = [
    "Nazali na mpasi ya ntolo makasi, nalingi lisungi",
    "Nazali na fiɛvɛrɛ moke, kasi nazali koyoka malamu",
    "Nzoto na ngai ezali kolemba makasi na nsima ya chimio",
    "Makila ezali kobima, nalingi mosungi ya mbango",
]

print("\n" + "="*60)
print("INFERENCE DEMO - Best Enhanced Model")
print("="*60)

# Use ensemble model for demo
for sentence in test_sentences:
    pred = ensemble_model.predict(sentence)
    print(f"\n📝 Text: {sentence}")
    print(f"🏷️  Prediction: {pred}")

print("\n" + "="*60)
print("✅ All done!")
print("="*60)


INFERENCE DEMO - Best Enhanced Model

📝 Text: Nazali na mpasi ya ntolo makasi, nalingi lisungi
🏷️  Prediction: Moderate

📝 Text: Nazali na fiɛvɛrɛ moke, kasi nazali koyoka malamu
🏷️  Prediction: Low

📝 Text: Nzoto na ngai ezali kolemba makasi na nsima ya chimio
🏷️  Prediction: Moderate

📝 Text: Makila ezali kobima, nalingi mosungi ya mbango
🏷️  Prediction: Low

✅ All done!


In [22]:
# ============================================================================
# SECTION 19: SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*70)
print("ENHANCED MODEL SUMMARY STATISTICS")
print("="*70)

print(f"\nTraining Data:")
print(f"  Total samples: {len(train_samples_aug)}")
print(f"  Emergency: {Counter(s['label'] for s in train_samples_aug)['Emergency']}")
print(f"  Moderate: {Counter(s['label'] for s in train_samples_aug)['Moderate']}")
print(f"  Low: {Counter(s['label'] for s in train_samples_aug)['Low']}")

print(f"\nTest Set Performance:")
for name, metrics in results.items():
    print(f"  {name}: Acc={metrics['accuracy']:.4f}, F1={metrics['macro_f1']:.4f}")

if external_df is not None:
    print(f"\nExternal Set Performance:")
    for name, metrics in external_results.items():
        print(f"  {name}: Acc={metrics['accuracy']:.4f}, F1={metrics['macro_f1']:.4f}")

print("\n" + "="*70)
print("✅ ENHANCED LINGALA MEDICAL TRIAGE SYSTEM COMPLETE")
print("="*70)


ENHANCED MODEL SUMMARY STATISTICS

Training Data:
  Total samples: 634
  Emergency: 245
  Moderate: 74
  Low: 315

Test Set Performance:
  SVM Baseline (char): Acc=0.9263, F1=0.8546
  XLM-R: Acc=0.9158, F1=0.8313
  Cross-Lingual: Acc=0.8632, F1=0.7450
  AfriBERTa: Acc=0.9684, F1=0.9188
  Ensemble: Acc=0.9684, F1=0.9188

External Set Performance:
  SVM Baseline: Acc=0.2600, F1=0.2195
  XLM-R: Acc=0.2100, F1=0.1317
  Cross-Lingual: Acc=0.2000, F1=0.1434
  AfriBERTa: Acc=0.2000, F1=0.1120
  Ensemble: Acc=0.2000, F1=0.1120

✅ ENHANCED LINGALA MEDICAL TRIAGE SYSTEM COMPLETE
